## Bing Song's HLA Task
Hi, so I'll need the HLA-I frequencies of about 20 populations from AFND, listed below:

- USA NMDP African American pop 2
- USA NMDP African
- USA NMDP Black South or Central American
- USA NMDP Caribean Black
- USA NMDP Chinese
- USA NMDP Filipino
- USA NMDP Japanese
- USA NMDP Korean
- USA NMDP South Asian Indian
- USA NMDP Southeast Asian
- USA NMDP Vietnamese
- USA NMDP Mexican or Chicano
- USA NMDP Hispanic South or Central American
- USA NMDP Caribean Hispanic
- USA NMDP European Caucasian
- USA NMDP Middle Eastern or North Coast of Africa
- USA NMDP North American Amerindian
- USA NMDP Alaska Native or Aleut
- USA NMDP American Indian South or Central America
- USA NMDP Hawaiian or other Pacific Islander
- USA NMDP Caribean Indian


For these, do the following for cleaning data:

Don't care about sample size.
Still check for the 2-, 4-, 6-, and 8-digit representations. The majority of the population should have standard 4-digit representations only. 
For 6- and 8-digit representations, first check if the same allele has 4-digit representations. If not, simply add up the frequencies like you did before. If it does, just use the 4-digit frequency and ignore the 6- and 8-digit ones.
Drop all 2-digit representations.

Remove the asterisk in allele names.
Check if the alleles are in the hla_embeddings_all.pkl file I shared earlier in our group chat. Drop those that are not in it.
Keep the original frequencies and do not normalize.
For each population, make a separate csv file of alleles and their corresponding frequencies. The filename should be the population name (with whitespaces replaced by underscores) like African_American_pop_2.csv and Alaska_Native_or_Aleut.csv

## My Plan 

Understand what is the hla_embeddings_all.pkl file. Print it out so I can understand it. 

Use the afnd.tsv file to get the HLA-I frequencies for those 20 populations and put into a pandas dataframe. 
Follow those steps above to get 4 digit representations only. 
Remove asterisk






In [1]:
import pandas as pd
import pickle
import re
import os

# ── 1. Load the HLA embeddings to know which alleles are valid ──────────────
with open("hla_embeddings_all.pkl", "rb") as f:
    hla_embeddings = pickle.load(f)

print(type(hla_embeddings))
if isinstance(hla_embeddings, dict):
    keys = list(hla_embeddings.keys())
    print(f"Number of keys: {len(keys)}")
    print("First 10 keys:", keys[:10])
elif hasattr(hla_embeddings, '__len__'):
    print(f"Length: {len(hla_embeddings)}")
    print("First 10 items:", list(hla_embeddings)[:10])
else:
    print(hla_embeddings)


<class 'dict'>
Number of keys: 2
First 10 keys: ['hlas', 'embeddings']


In [2]:
# Inspect the structure more
print("Keys in pkl:", list(hla_embeddings.keys()))
hlas_list = hla_embeddings['hlas']
print(f"Number of HLA entries: {len(hlas_list)}")
print("Type of hlas:", type(hlas_list))
print("First 10 HLA entries:", hlas_list[:10])
print("Embeddings shape:", hla_embeddings['embeddings'].shape if hasattr(hla_embeddings['embeddings'], 'shape') else type(hla_embeddings['embeddings']))


Keys in pkl: ['hlas', 'embeddings']
Number of HLA entries: 8722
Type of hlas: <class 'numpy.ndarray'>
First 10 HLA entries: ['A01:01' 'A01:02' 'A01:03' 'A01:06' 'A01:07' 'A01:08' 'A01:09' 'A01:10'
 'A01:100' 'A01:101']
Embeddings shape: torch.Size([8722, 32])


In [3]:
# ── 2. Explore the afnd.tsv file ───────────────────────────────────────────
df_raw = pd.read_csv("afnd.tsv", sep="\t")
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head(5)


Shape: (152716, 7)
Columns: ['group', 'gene', 'allele', 'population', 'indivs_over_n', 'alleles_over_2n', 'n']


,group,gene,allele,population,indivs_over_n,alleles_over_2n,n
0,cyt,RANTES-,RANTES/ -109 CC,England Northwest Cytokine,0.0,NaN,500
1,cyt,RANTES-,RANTES/ -109 CT,England Northwest Cytokine,11.5,NaN,500
2,cyt,RANTES-,RANTES/ -109 TT,England Northwest Cytokine,88.5,NaN,500
3,cyt,PDGF-B-,PDGF B/ 1135 AA,England Northwest Cytokine,50.9,NaN,500
4,cyt,PDGF-B-,PDGF B/ 1135 AC,England Northwest Cytokine,36.8,NaN,500


In [4]:
# Check what populations exist matching "NMDP"
nmdp_pops = df_raw[df_raw['population'].str.contains('NMDP', na=False)]['population'].unique()
print(f"Total NMDP populations: {len(nmdp_pops)}")
for p in sorted(nmdp_pops):
    print(repr(p))


Total NMDP populations: 22
'USA Caucasian NMDP KIR'
'USA NMDP African'
'USA NMDP African American pop 2'
'USA NMDP Alaska Native or Aleut'
'USA NMDP American Indian South or Central America'
'USA NMDP Black South or Central American'
'USA NMDP Caribean Black'
'USA NMDP Caribean Hispanic'
'USA NMDP Caribean Indian'
'USA NMDP Chinese'
'USA NMDP European Caucasian'
'USA NMDP Filipino'
'USA NMDP Hawaiian or other Pacific Islander'
'USA NMDP Hispanic South or Central American'
'USA NMDP Japanese'
'USA NMDP Korean'
'USA NMDP Mexican or Chicano'
'USA NMDP Middle Eastern or North Coast of Africa'
'USA NMDP North American Amerindian'
'USA NMDP South Asian Indian'
'USA NMDP Southeast Asian'
'USA NMDP Vietnamese'


In [5]:
# Check HLA-I genes and example allele formats in NMDP populations
target_pops = [
    'USA NMDP African American pop 2',
    'USA NMDP African',
    'USA NMDP Black South or Central American',
    'USA NMDP Caribean Black',
    'USA NMDP Chinese',
    'USA NMDP Filipino',
    'USA NMDP Japanese',
    'USA NMDP Korean',
    'USA NMDP South Asian Indian',
    'USA NMDP Southeast Asian',
    'USA NMDP Vietnamese',
    'USA NMDP Mexican or Chicano',
    'USA NMDP Hispanic South or Central American',
    'USA NMDP Caribean Hispanic',
    'USA NMDP European Caucasian',
    'USA NMDP Middle Eastern or North Coast of Africa',
    'USA NMDP North American Amerindian',
    'USA NMDP Alaska Native or Aleut',
    'USA NMDP American Indian South or Central America',
    'USA NMDP Hawaiian or other Pacific Islander',
    'USA NMDP Caribean Indian',
]

df_nmdp = df_raw[df_raw['population'].isin(target_pops)].copy()
print("Shape of NMDP subset:", df_nmdp.shape)
print("\nUnique genes:", df_nmdp['gene'].unique())
print("\nGroups:", df_nmdp['group'].unique())


Shape of NMDP subset: (10398, 7)

Unique genes: ['DPB1' 'DQB1' 'DRB1' 'B' 'A' 'C']

Groups: ['hla']


In [6]:
# HLA-I genes are A, B, C. Check sample allele formats
df_hla1 = df_nmdp[df_nmdp['gene'].isin(['A', 'B', 'C'])].copy()
print("HLA-I rows:", len(df_hla1))
print("\nSample alleles:")
print(df_hla1['allele'].head(20).tolist())
print("\nSample of alleles_over_2n (frequency column):")
print(df_hla1[['allele', 'alleles_over_2n', 'population']].head(10).to_string())


HLA-I rows: 8003

Sample alleles:
['B*52', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02', 'B*07:02']

Sample of alleles_over_2n (frequency column):
        allele alleles_over_2n                                         population
55401     B*52          0.0001           USA NMDP Black South or Central American
59353  B*07:02          0.0689                                   USA NMDP African
59354  B*07:02          0.0729                    USA NMDP African American pop 2
59355  B*07:02          0.0780                    USA NMDP Alaska Native or Aleut
59356  B*07:02          0.0960  USA NMDP American Indian South or Central America
59357  B*07:02          0.0729                            USA NMDP Caribean Black
59358  B*07:02          0.0568                         USA NMDP Caribean Hispanic
59359  B*07:02          0.0510               

In [7]:
# Understand digit-resolution of alleles
# Count colons to determine resolution
df_hla1['colon_count'] = df_hla1['allele'].str.count(':')
df_hla1['digit_class'] = df_hla1['colon_count'].map({0: '2-digit', 1: '4-digit', 2: '6-digit', 3: '8-digit'})
print(df_hla1['digit_class'].value_counts())
print("\nExamples of 2-digit:", df_hla1[df_hla1['digit_class']=='2-digit']['allele'].unique()[:10])
print("Examples of 6-digit:", df_hla1[df_hla1['digit_class']=='6-digit']['allele'].unique()[:5])
print("Examples of 8-digit:", df_hla1[df_hla1['digit_class']=='8-digit']['allele'].unique()[:5])


digit_class
4-digit    7918
6-digit      76
8-digit       5
2-digit       4
Name: count, dtype: int64

Examples of 2-digit: ['B*52' 'A*30' 'A*32' 'A*36']
Examples of 6-digit: ['B*07:02:31' 'B*07:02:36' 'B*07:05:03' 'B*07:05:05' 'B*07:68:01']
Examples of 8-digit: ['A*29:01:01:02N' 'A*68:01:01:01' 'A*68:02:01:01' 'A*68:02:01:03'
 'C*08:02:01:01']


In [8]:
# ── 3. Full Cleaning Pipeline ───────────────────────────────────────────────
# Build a set of valid allele names from the embeddings pkl (no asterisk format: e.g. "A01:01")
valid_alleles = set(hla_embeddings['hlas'])
print(f"Valid alleles in pkl: {len(valid_alleles)}")
print("Sample:", list(valid_alleles)[:5])


Valid alleles in pkl: 8722
Sample: [np.str_('C04:249'), np.str_('B55:60'), np.str_('B55:15'), np.str_('B15:386'), np.str_('A02:492')]


In [9]:
# Check: AFND allele "B*07:02" → remove asterisk → "B07:02"
# PKL has "B07:02"? Let's verify
test = "B*07:02".replace("*", "")
print("Transformed:", test)
print("In valid_alleles?", test in valid_alleles)

# Also check a few more
for a in ["A*02:01", "C*07:02", "B*57:01"]:
    t = a.replace("*", "")
    print(f"{a} -> {t}: in pkl? {t in valid_alleles}")


Transformed: B07:02
In valid_alleles? True
A*02:01 -> A02:01: in pkl? True
C*07:02 -> C07:02: in pkl? True
B*57:01 -> B57:01: in pkl? True


In [10]:
# ── 4. Clean & collapse to 4-digit using utils.py ──────────────────────────
import importlib, utils
importlib.reload(utils)
from utils import clean_data, collapse_to_4digit

# Filter raw data for the 21 target populations
df_target = df_raw[df_raw['population'].isin(target_pops)].copy()

# Basic clean: keep HLA group, Class I genes (A, B, C), remove G-groups
df_cleaned = clean_data(df_target, class1_only=True, remove_g_groups=True, verbose=True)


Starting shape: (10398, 7)
After filtering for HLA: (10398, 7)
After filtering for Class I (A, B, C): (8003, 7)
After removing 0 G-group rows: (8003, 8)
Final shape after dropping columns: (8003, 5)


In [13]:
# ── 5. Attempt collapse_to_4digit ──────────────────────────────────────────
from utils import collapse_to_4digit

df_4digit = collapse_to_4digit(df_cleaned, verbose=True)


Starting collapse_to_4digit
Input shape: (8003, 5)
Input studies: 21

Resolution distribution before collapse:
{'4-digit': 7918, '6-digit': 76, '8-digit': 5, '2-digit': 4}

--- Step 1: Collapse 8-digit → 6-digit alleles ---


Collapsing 8-digit to 6-digit:   0%|          | 0/21 [00:00<?, ?it/s]

/home/hongj/allelefreq/utils.py:197: FutureWarning: Operation between Series with different indexes that are not of numpy boolean or object dtype will no longer return a numpy boolean result in a future version. Cast both Series to object type to maintain the prior behavior.
  children_mask = (pop_mask) & (df_result['parent_6digit'] == parent_6d)
/home/hongj/allelefreq/utils.py:201: FutureWarning: Operation between Series with different indexes that are not of numpy boolean or object dtype will no longer return a numpy boolean result in a future version. Cast both Series to object type to maintain the prior behavior.
  parent_mask = (pop_mask) & (df_result['allele'] == parent_6d)
/home/hongj/allelefreq/utils.py:222: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  template_idx = df_result[children_mask].index[0]
Collapsing 8-digit to 6-digit:   0%|          | 0/21 [00:00<?, ?it/s]


IndexError: index 0 is out of bounds for axis 0 with size 0

In [22]:
# ── 6. Fixed collapse pipeline — two-pass (8→6, then 6→4) ──────────────────
# Correct rules for each pass:
#   - If a parent at lower resolution EXISTS → keep parent freq as-is, discard children
#   - If NO parent exists → sum children freqs into a new parent-resolution entry
# (Never take max — parent freq wins if it exists, sum only when there's no parent)

from utils import get_allele_resolution, extract_allele_parts

def collapse_pass(df, from_res, to_res, parent_key):
    """
    One collapse pass: fold `from_res` alleles into `to_res` parents, per population.
      - parent exists → keep parent freq unchanged, discard children
      - parent absent  → create parent with freq = sum(children)
    """
    rows_to_add = []

    for pop in df['population'].unique():
        pop_df = df[df['population'] == pop]
        existing_parent = set(
            pop_df[pop_df['resolution'] == to_res]['allele']
        )
        children = pop_df[pop_df['resolution'] == from_res]

        # Group children by their parent prefix
        prefix_groups = {}
        for _, row in children.iterrows():
            parts = extract_allele_parts(row['allele'])
            if parts and parts[parent_key]:
                prefix_groups.setdefault(parts[parent_key], []).append(row)

        for parent_allele, child_rows in prefix_groups.items():
            if parent_allele in existing_parent:
                # Parent exists → keep its freq, children will be dropped
                pass
            else:
                # No parent → create one by summing children freqs
                freq_sum = sum(r['alleles_over_2n'] for r in child_rows)
                new_row = child_rows[0].copy()
                new_row['allele'] = parent_allele
                new_row['resolution'] = to_res
                new_row['alleles_over_2n'] = freq_sum
                rows_to_add.append(new_row)

    if rows_to_add:
        df = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)

    # Drop the from_res rows (both those with parents and orphans now promoted)
    df = df[df['resolution'] != from_res].copy()
    return df


def collapse_higher_to_4digit(df):
    df_result = df.copy().reset_index(drop=True)
    df_result['resolution'] = df_result['allele'].apply(get_allele_resolution)

    print("Resolution distribution before:")
    print(df_result['resolution'].value_counts().to_dict())

    # Pass 1: collapse 8-digit → 6-digit
    df_result = collapse_pass(df_result, from_res='8-digit', to_res='6-digit', parent_key='6digit')
    print("\nAfter 8→6 pass:", df_result['resolution'].value_counts().to_dict())

    # Pass 2: collapse 6-digit → 4-digit
    df_result = collapse_pass(df_result, from_res='6-digit', to_res='4-digit', parent_key='4digit')
    print("After 6→4 pass:", df_result['resolution'].value_counts().to_dict())

    # Drop 2-digit entries
    df_result = df_result[df_result['resolution'] == '4-digit'].copy()
    df_result = df_result.drop(columns=['resolution']).reset_index(drop=True)
    return df_result


df_4digit = collapse_higher_to_4digit(df_cleaned)
print("\nFinal shape:", df_4digit.shape)
print("Populations:", df_4digit['population'].nunique())
print(df_4digit.head())


Resolution distribution before:
{'4-digit': 7918, '6-digit': 76, '8-digit': 5, '2-digit': 4}

After 8→6 pass: {'4-digit': 7918, '6-digit': 80, '2-digit': 4}
After 6→4 pass: {'4-digit': 7967, '2-digit': 4}

Final shape: (7967, 5)
Populations: 21
  gene   allele                                         population  \
0    B  B*07:02                                   USA NMDP African   
1    B  B*07:02                    USA NMDP African American pop 2   
2    B  B*07:02                    USA NMDP Alaska Native or Aleut   
3    B  B*07:02  USA NMDP American Indian South or Central America   
4    B  B*07:02                            USA NMDP Caribean Black   

   alleles_over_2n       n  
0           0.0689   28557  
1           0.0729  416581  
2           0.0780    1376  
3           0.0960    5926  
4           0.0729   33328  


In [23]:
# ── 7. Remove asterisk, filter against embeddings, export CSVs ──────────────
import os

output_dir = "bingsong_output"
os.makedirs(output_dir, exist_ok=True)

# valid_alleles from pkl is in format "A02:01" (no asterisk)
# AFND alleles are "A*02:01" — strip asterisk to match
df_export = df_4digit.copy()
df_export['allele_clean'] = df_export['allele'].str.replace('*', '', regex=False)

# How many alleles are in the embeddings?
total = len(df_export)
in_pkl = df_export['allele_clean'].isin(valid_alleles).sum()
print(f"Entries in cleaned dataframe with alleles in pkl: {in_pkl}/{total} ({in_pkl/total*100:.1f}%)")
print("Dropped entries:", total - in_pkl)

# Drop alleles not in embeddings
df_export = df_export[df_export['allele_clean'].isin(valid_alleles)].copy()

# Final columns: allele (clean, no asterisk) and frequency
df_export = df_export[['allele_clean', 'alleles_over_2n', 'population']].rename(
    columns={'allele_clean': 'allele', 'alleles_over_2n': 'frequency'}
)

# Export one CSV per population
def pop_to_filename(pop_name):
    # Remove "USA NMDP " prefix, replace spaces with underscores
    name = pop_name.replace('USA NMDP ', '').strip()
    return name.replace(' ', '_') + '.csv'

print("Saving df to csvs:")
saved = []
for pop, group in df_export.groupby('population'):
    fname = pop_to_filename(pop)
    fpath = os.path.join(output_dir, fname)
    group[['allele', 'frequency']].to_csv(fpath, index=False)
    saved.append((fname, len(group)))
    print(f"  {fname}: {len(group)} alleles")

print(f"\nSaved {len(saved)} files to '{output_dir}/'")


Entries in cleaned dataframe with alleles in pkl: 7894/7967 (99.1%)
Dropped entries: 73
Saving df to csvs:
  African.csv: 354 alleles
  African_American_pop_2.csv: 638 alleles
  Alaska_Native_or_Aleut.csv: 112 alleles
  American_Indian_South_or_Central_America.csv: 136 alleles
  Black_South_or_Central_American.csv: 213 alleles
  Caribean_Black.csv: 205 alleles
  Caribean_Hispanic.csv: 444 alleles
  Caribean_Indian.csv: 151 alleles
  Chinese.csv: 413 alleles
  European_Caucasian.csv: 998 alleles
  Filipino.csv: 332 alleles
  Hawaiian_or_other_Pacific_Islander.csv: 284 alleles
  Hispanic_South_or_Central_American.csv: 608 alleles
  Japanese.csv: 165 alleles
  Korean.csv: 350 alleles
  Mexican_or_Chicano.csv: 586 alleles
  Middle_Eastern_or_North_Coast_of_Africa.csv: 508 alleles
  North_American_Amerindian.csv: 345 alleles
  South_Asian_Indian.csv: 462 alleles
  Southeast_Asian.csv: 308 alleles
  Vietnamese.csv: 282 alleles

Saved 21 files to 'bingsong_output/'


In [16]:
# ── 8. Inspect dropped alleles & check African American pop 2 genes ─────────

df_export_all = df_4digit.copy()
df_export_all['allele_clean'] = df_export_all['allele'].str.replace('*', '', regex=False)
df_export_all['in_pkl'] = df_export_all['allele_clean'].isin(valid_alleles)

# Show dropped alleles
dropped = df_export_all[~df_export_all['in_pkl']][['gene', 'allele', 'allele_clean', 'alleles_over_2n', 'population']]
print(f"=== Dropped alleles (not in pkl): {len(dropped)} ===")
print(dropped.to_string())

# Check African American pop 2 gene distribution
print("\n=== African American pop 2 — genes present ===")
aa2 = df_export_all[df_export_all['population'] == 'USA NMDP African American pop 2']
print("Before pkl filter:", aa2['gene'].value_counts().to_dict())
aa2_dropped = aa2[~aa2['in_pkl']]
print(f"Dropped from AA pop 2: {len(aa2_dropped)}")
if len(aa2_dropped):
    print(aa2_dropped[['gene', 'allele', 'allele_clean']].to_string())


=== Dropped alleles (not in pkl): 73 ===
     gene     allele allele_clean  alleles_over_2n                                        population
197     B  B*07:111N     B07:111N     6.000000e-06                                   USA NMDP Korean
218     B  B*07:161N     B07:161N     2.000000e-04          USA NMDP Black South or Central American
811     B   B*15:26N      B15:26N     3.300000e-05          USA NMDP Black South or Central American
812     B   B*15:26N      B15:26N     3.000000e-06       USA NMDP Hispanic South or Central American
1052    B   B*15:79N      B15:79N     4.000000e-07                       USA NMDP European Caucasian
1158    B  B*15:209N     B15:209N     1.000000e-05                                  USA NMDP African
1165    B  B*15:294N     B15:294N     1.900000e-05          USA NMDP Black South or Central American
1171    B  B*15:302N     B15:302N     1.900000e-05          USA NMDP Black South or Central American
1173    B  B*15:304N     B15:304N     1.900000e-05

## Summary
- Use same clean function
- Use a different collapse_to_4digit() function due to his requirements. 
- Don't need to run clean_and_normalize() after collapsing due to requirements. 
- All the dropped entries due to not being in hla_embeddings_all.pkl is due to having a N or Q at the end of the allele names
- Results saved in `./bingsong_output/`
